# Diseño de un Pipeline ETL en Python 🐍🔧

¡Hola, equipo! 👋 En esta sesión vamos a construir nuestro primer **pipeline de ETL (Extract, Transform, Load)** en Python. Es una de las tareas más comunes y fundamentales en el mundo de los datos.

> 🧠 **Analogía:** Piensa en un ETL como el proceso de **cocinar**. Primero, **extraes** los ingredientes (datos) del supermercado (API, base de datos, etc.). Luego, los **transformas**: los lavas, cortas y preparas. Finalmente, los **cargas** en un plato para servirlos (una base de datos, un dashboard).

## Objetivos del Webinar 🎯

1.  **Explicar qué es ETL** (y cuándo usarlo).
2.  **Extraer (E)** datos desde el API de OpenAQ.
3.  **Transformar (T)** aplicando limpieza, cambios de tipo y validaciones.
4.  **Crear tablas de resumen** (nuestra capa de "oro" o `gold_data`).
5.  **Cargar (L)** los datos en una base de datos SQLite.
6.  **Ensamblar un pipeline** simple y reutilizable con funciones.

## 🏛️ ¿Qué es ETL y por qué es tan importante?

**ETL** significa **Extract, Transform, Load** (Extraer, Transformar, Cargar). Es el proceso que usamos para mover datos desde una o varias fuentes, limpiarlos, darles una estructura útil y almacenarlos en un destino final, como una base de datos o un Data Warehouse.

-   **Extract (Extraer):** Obtener los datos desde su origen.
-   **Transform (Transformar):** Limpiar, validar, enriquecer y modelar los datos. ¡Aquí es donde ocurre la magia! ✨
-   **Load (Cargar):** Guardar los datos transformados en su destino final.

Usamos ETL para consolidar datos, prepararlos para análisis, alimentar dashboards o para cualquier tarea que requiera datos limpios y estructurados.

## 1. Extract: Extrayendo datos del API de OpenAQ 🌍

Vamos a usar el API de [OpenAQ](https://openaq.org/) para obtener datos de calidad del aire. Ya vimos cómo conectarnos en sesiones anteriores, ¡así que vamos a aplicar lo aprendido!

Nuestro objetivo será extraer las mediciones de `pm25` para un país específico en un rango de fechas.

In [ ]:
import requests
import pandas as pd
from datetime import datetime


In [ ]:
API_KEY=""

In [ ]:
def get_country_data():
    url = "https://api.openaq.org/v3/countries"
    headers = {
        "X-API-Key": API_KEY
    }

    params={'limit':1000}
    response = requests.get(url, headers=headers,params=params)
    response.raise_for_status()
    return response.json()


def get_locations_data_by_country_ids(country_ids):
    url = "https://api.openaq.org/v3/locations"


    headers = {
        "X-API-Key": API_KEY
    }

    params={'limit':1000,'countries_id':country_ids}
    response = requests.get(url, headers=headers,params=params)
    response.raise_for_status()

    data = response.json()

    return data


def process_locations_sensor_data(location_data):
    locations_sensors=pd.DataFrame(location_data['results'])
    locations_sensors=locations_sensors.explode('sensors')
    sensor_data=pd.json_normalize(locations_sensors['sensors']).reset_index(drop=True)
    locations_sensors=locations_sensors[['id','name','locality']].reset_index(drop=True)
    locations_sensors.rename(columns={'id':'id_location','name':'name_location'},inplace=True)
    locations_sensors = pd.concat([locations_sensors,sensor_data],axis=1)
    return locations_sensors


def get_daily_measurements_by_sensor_id(sensor_id,params):

    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements/daily"
    headers = {
        "X-API-Key": API_KEY
    }
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()

    return response.json()['results']



### Extracción de los datos 

In [ ]:
data= get_country_data()
countries=pd.DataFrame(data['results'])
countries.query('code == "MX"')

In [ ]:
#Params
CODE_COUNTRY=157

In [ ]:
locations_data= get_locations_data_by_country_ids(157)
locations_data = process_locations_sensor_data(locations_data)

In [ ]:
locations_data

In [ ]:
locations_data.query('locality == "GUANAJUATO" and name == "pm25 µg/m³"')

In [ ]:
ids_selection=locations_data.query('locality == "GUANAJUATO" and name == "co ppm"')['id'].to_list()

In [ ]:
locations_data[['name_location','id_location','locality','id']]

In [ ]:
raw_data =[]
params = {
    "datetime_from": "2025-01-01T00:00:00Z",
    "datetime_to": "2025-01-31T23:59:59Z",
    "limit": 31
}
for sensor_id in ids_selection:
    try:
        current_data=get_daily_measurements_by_sensor_id(sensor_id,params)
        raw_data.append({'sensor_id':sensor_id,'data':current_data})
    except Exception as ee:
        print("Failed")
        print (ee)
        pass


## 2. Transform: Limpieza y modelado de datos 🧹

Los datos crudos rara vez son perfectos. La etapa de transformación es crucial para asegurar la calidad.

Nuestras tareas serán:
- Convertir la lista de diccionarios a un DataFrame de pandas.
- Seleccionar solo las columnas que nos interesan.
- Convertir la fecha a un formato `datetime`.
- Desanidar la información .
- Renombrar columnas para que sean más claras.
- Validar que los valores de `pm25` no sean negativos.

In [ ]:
def cleaning_data_sensor(raw_data):
    df=pd.DataFrame(raw_data) 
    df=df.explode('data')
    df = pd.concat([df[['sensor_id']].reset_index(drop=True),pd.json_normalize(df['data']).reset_index(drop=True)],axis=1)
    df['date_at']=pd.to_datetime(df['period.datetimeFrom.local'])
    df.rename(columns={'parameter.name':'parameter'},inplace=True)
    df=df[['date_at','sensor_id','parameter','value']]
    return df[df['value']>=0]


In [ ]:
locations_data=locations_data[['id','name_location','locality']].rename(columns={'id':'sensor_id'})
cleaning_measures_data=cleaning_data_sensor(raw_data)

In [ ]:
df_silver=locations_data.merge(cleaning_measures_data)

In [ ]:
df_silver

> En la industria, a esta capa de datos limpios y estructurados a menudo se le llama **"Silver Layer"** (Capa de Plata).

## 3. Creando la capa "Gold": Tablas de resumen 🏆

La capa "Gold" (Oro) contiene datos agregados y listos para el negocio. Son las tablas que un analista o un modelo de Machine Learning consumiría directamente.

Vamos a crear una tabla que resuma el promedio diario de `pm25` por ciudad.

In [ ]:
def create_gold_data(df):
    """
    Crea una tabla de resumen (capa Gold) con el promedio diario de pm25 por ciudad.
    """
    if df.empty:
        return pd.DataFrame()

    print("🏆 Creando la tabla de resumen (Gold)...")
    
    df['date'] = df['date_at'].dt.date
    df=df.rename(columns={'locality':'city'})
    
    df_gold = df.groupby(['city', 'date']).agg(
        avg_pm25=('value', 'mean'),
        max_pm25=('value', 'max'),
        min_pm25=('value', 'min'),
        num_measurements=('value', 'count')
    ).reset_index()
    
    df_gold['avg_pm25'] = df_gold['avg_pm25'].round(2)
    
    print("✨ ¡Tabla Gold creada exitosamente!")
    return df_gold

In [ ]:
# Creemos nuestra tabla de oro
df_gold = create_gold_data(df_silver.copy()) # Usamos .copy() para evitar warnings
if not df_gold.empty:
    print(df_gold.head())

## 4. Load: Cargando los datos a una Base de Datos SQLite 💾

¡Es hora de guardar nuestro trabajo! Usaremos **SQLite**, una base de datos súper ligera que guarda todo en un solo archivo. Es perfecta para proyectos como este.

Pandas hace que este proceso sea increíblemente fácil con el método `.to_sql()`.

In [ ]:
import sqlite3

def load_to_sqlite(tables, db_name='openaq.db'):
    """
    Carga un diccionario de DataFrames a tablas en una base de datos SQLite.
    - tables: {'nombre_tabla': DataFrame}
    """
    print(f"💾 Cargando datos a la base de datos '{db_name}'...")
    
    try:
        conn = sqlite3.connect(db_name)
        
        for table_name, df in tables.items():
            if not df.empty:
                df.to_sql(table_name, conn, if_exists='replace', index=False)
                print(f"  - Tabla '{table_name}' cargada con {len(df)} filas.")
        
        conn.close()
        print("✅ Carga finalizada.")
        
    except sqlite3.Error as e:
        print(f"🔥 Error al cargar a SQLite: {e}")

In [ ]:
# Preparamos el diccionario de tablas y ejecutamos la carga
tables_to_load = {
    'raw_measurements': cleaning_measures_data, # Guardamos también la data cruda por si acaso
    'silver_measurements': df_silver,
    'gold_daily_summary': df_gold
}

load_to_sqlite(tables_to_load)

### Verificación

¿Cómo sabemos que funcionó? ¡Leamos los datos de vuelta desde la base de datos!

In [ ]:
def run_query_sqlite(db_path,query):
  query_parsed= query.replace('\xa0', ' ').replace('\n',' ').strip()
  conn = sqlite3.connect(db_path)
  df = pd.read_sql(query_parsed, conn)
  conn.close()
  return df

In [ ]:
def verify_load(db_name='openaq.db'):
    conn = sqlite3.connect(db_name)
    
    try:
        print("\n🔍 Verificando datos en la base de datos:")
        # Leemos los nombres de las tablas
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
        print("Tablas encontradas:", tables['name'].tolist())

        # Leemos algunas filas de la tabla gold
        df_check = pd.read_sql("SELECT * FROM gold_daily_summary LIMIT 5", conn)
        print("\nPrimeras 5 filas de 'gold_daily_summary':")
        print(df_check)
        
    finally:
        conn.close()

verify_load()

## 5. ¡Juntando todo en un Pipeline! 🚀

Hemos creado funciones para cada paso. Ahora, podemos unirlas en un solo script que ejecute todo el proceso de forma ordenada.

In [ ]:
def run_openaq_etl_pipeline(params):
    """
    Ejecuta el pipeline ETL completo para OpenAQ.
    """
    country_code=params['country_code']
    cities = params['cities']
    chemical = params['param']
    print("=============================================")
    print(f"🚀 INICIANDO PIPELINE ETL PARA {country_code} 🚀")
    print("=============================================")

    countries_data = get_country_data()
    countries=pd.DataFrame(countries_data['results'])
    country_id= countries.query('code == @country_code').id.to_list()[0]
    locations_data= get_locations_data_by_country_ids(country_id)
    locations_data = process_locations_sensor_data(locations_data)
    locations_data_selection =locations_data[['name_location','id_location','locality','id']].rename(columns={'id':'sensor_id'})
    ids_selection=locations_data.query('locality in @cities and name == @chemical')['id'].to_list()

    params = {
    "datetime_from": params["datetime_from"],
    "datetime_to": params["datetime_to"],
    "limit": 31
    }
    raw_data_measumerent =[]
    for sensor_id in ids_selection:
        try:
            current_data=get_daily_measurements_by_sensor_id(sensor_id,params)
            raw_data_measumerent.append({'sensor_id':sensor_id,'data':current_data})
        except Exception as ee:
            print("Failed")
            print (ee)
            pass
    
    cleaning_measures_data=cleaning_data_sensor(raw_data_measumerent)
    df_silver=locations_data_selection.merge(cleaning_measures_data)
    df_gold = create_gold_data(df_silver.copy())
    tables_to_load = {
    'raw_measurements': cleaning_measures_data, # Guardamos también la data cruda por si acaso
    'silver_measurements': df_silver,
    'gold_daily_summary': df_gold
    }
    load_to_sqlite(tables_to_load)



    

In [ ]:
# ¡Ejecutemos todo el flujo con una sola llamada!
params =dict(country_code='MX', datetime_from='2024-03-01', datetime_to='2024-03-02',cities=['GUANAJUATO'],param= "pm25 µg/m³")
run_openaq_etl_pipeline(params)

¡Y ahí lo tienes! Un pipeline de ETL funcional. Este es el pilar para construir sistemas de datos robustos y automatizados.

## Conclusión y Próximos Pasos 🏁

Hoy aprendimos a:
- Estructurar un proceso ETL con funciones claras.
- Manejar datos desde una API hasta una base de datos.
- Diferenciar entre capas de datos (Raw, Silver, Gold).
- Usar SQLite para almacenar nuestros resultados de forma sencilla.

**Para seguir aprendiendo:**
- **Mejora el pipeline:** Añade manejo de errores más robusto, logs para cada paso o un sistema de configuración.
- **Automatízalo:** Investiga cómo programar la ejecución de este script una vez al día (¡hola, `cron` o Airflow!).
- **Expande las transformaciones:** ¿Qué otras métricas podrías calcular en la capa Gold?

¡Excelente trabajo, equipo! 💪✨